# Build Object B (orthologue gene space)

* **Developed by:** Anna Maguza
* **Affilation:** CellZome, a GSK company
* **Created date:** 2026-05-08
* **Last modified date:** 2026-05-17

Map mouse SC datasets to human via 1:1 Ensembl orthologues, concat with Object A.


## Goal

Build **Object B (orthologue gene space)**: concat Object A with all mouse single-cell LGR5 datasets after mapping mouse symbols → human symbols via the 1:1 Ensembl orthologue table (notebook 1b). Save as `object_b_cross_species_orth.h5ad`.

Strategy: each mouse file is mapped to human symbols, then re-indexed onto the **full 1:1 orthologue space** (zeros for missing genes). Final concat is outer join with Object A so we don't lose all genes whenever any one platform misses one.

In [1]:
import os, sys, gc, json, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.verbosity = 2
sc.settings.dpi = 120
sc.settings.dpi_save = 300
plt.rcParams.update({
    'savefig.bbox': 'tight', 'savefig.dpi': 300, 'figure.dpi': 120,
    'font.family': ['Arial', 'Helvetica', 'DejaVu Sans'], 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

REPO         = Path('/Users/am336941/Library/CloudStorage/OneDrive-GSK/Desktop/Fetal_stem_cells_analysis')
DATA_OUT     = Path('/Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced')
ATLAS_PATH   = Path('/Users/am336941/PhD/data/gut_data/gut_hs_all_datasets_full_annotated_AM_30102025_181544_raw.h5ad')
LGR5_DIR     = REPO / 'data' / 'LGR5_analysis'
ORTH_PATH    = Path('/Users/am336941/PhD/data/LGR5_analysis_data/human_mouse_orthologues_ensembl.txt')
DATA_OUT.mkdir(parents=True, exist_ok=True)

def stamp() -> str:
    return dt.datetime.now().strftime('%d%m%Y_%H%M%S')


/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
import re
obj_a = sc.read_h5ad(DATA_OUT / 'object_a_human.h5ad')

# de-duplicate std files: keep latest timestamp per study
all_std = sorted(DATA_OUT.glob('gut_mm_*_std_*_raw.h5ad'))
by_study = {}
for f in all_std:
    m = re.match(r'gut_mm_(.+?)_std_(\d+_\d+)_raw\.h5ad', f.name)
    if m: by_study.setdefault(m.group(1), []).append(f)
mm_files = [sorted(v)[-1] for v in by_study.values()]
print(f'{len(mm_files)} mouse SC files (latest per study):')
for f in mm_files: print(' -', f.name)


7 mouse SC files (latest per study):
 - gut_mm_GSE145866_std_17052026_232918_raw.h5ad
 - gut_mm_GSE62270_std_17052026_232920_raw.h5ad
 - gut_mm_GSE62784_std_17052026_232920_raw.h5ad
 - gut_mm_GSE92865_std_17052026_232920_raw.h5ad
 - gut_mm_GSE99457_std_17052026_232923_raw.h5ad
 - gut_mm_Grün2016_std_17052026_232925_raw.h5ad
 - gut_mm_Haber2017_std_17052026_232925_raw.h5ad


## Map each mouse file to human symbols on the full orthologue space

In [3]:
orth = pd.read_csv(DATA_OUT / 'orthologues_one_to_one.csv', index_col=0)['human_symbol']
print('1:1 orthologues:', len(orth))
ORTH_GENES = sorted(set(orth.values))
print('unique human gene targets:', len(ORTH_GENES))

def normalise_var_names(a):
    if a.var_names.astype(str).str.fullmatch(r'\d+').all():
        for col in ['gene_name', 'gene_symbols', 'feature_name', 'symbol', 'Gene']:
            if col in a.var.columns:
                a.var_names = a.var[col].astype(str).values
                break
    a.var_names_make_unique()
    return a


def _x_is_raw(adata) -> bool:
    s = adata.X.sum(axis=0)
    if hasattr(s, 'A1'):
        s = s.A1
    s = np.asarray(s).ravel()
    return np.array_equal(s.astype(int), s)


mapped = []
skipped_nonraw = []
for f in mm_files:
    a = sc.read_h5ad(f)
    if not _x_is_raw(a):
        print(f'SKIP {f.name}: .X is not integer counts (Object B must contain only raw data)')
        skipped_nonraw.append(f.name)
        continue
    a = normalise_var_names(a)
    keep = a.var_names.isin(orth.index)
    a = a[:, keep].copy()
    a.var_names = pd.Index(orth.loc[a.var_names].values)
    a.var_names_make_unique()
    a.obs['species'] = 'mouse'
    if 'sample_id' not in a.obs.columns:
        a.obs['sample_id'] = a.obs.get('sample', 'unknown').astype(str)
    mapped.append(a)
    print(f.name, '→', a.shape)

if skipped_nonraw:
    print('\nstudies excluded from Object B (non-integer .X):')
    for s in skipped_nonraw:
        print(' -', s)


1:1 orthologues: 17045
unique human gene targets: 17045


gut_mm_GSE145866_std_17052026_232918_raw.h5ad → (4815, 15145)
SKIP gut_mm_GSE62270_std_17052026_232920_raw.h5ad: .X is not integer counts (Object B must contain only raw data)
gut_mm_GSE62784_std_17052026_232920_raw.h5ad → (16, 13386)


gut_mm_GSE92865_std_17052026_232920_raw.h5ad → (13247, 15145)


gut_mm_GSE99457_std_17052026_232923_raw.h5ad → (5570, 15145)
gut_mm_Grün2016_std_17052026_232925_raw.h5ad → (96, 17045)


gut_mm_Haber2017_std_17052026_232925_raw.h5ad → (1453, 15860)

studies excluded from Object B (non-integer .X):
 - gut_mm_GSE62270_std_17052026_232920_raw.h5ad


## Concat with Object A using outer join (zeros for missing genes)

In [4]:
all_objs = {'object_a': obj_a, **{m.obs['Study_name'].iloc[0] if 'Study_name' in m.obs.columns else f'mm_{i}': m
                                  for i, m in enumerate(mapped)}}
big = ad.concat(all_objs, axis=0, join='outer', label='source_b', merge='unique', fill_value=0)
big.obs_names_make_unique()
print('Object B (orth):', big.shape)
print(big.obs['source_b'].value_counts())

out = DATA_OUT / 'object_b_cross_species_orth.h5ad'
big.write_h5ad(out, compression='gzip')
print('saved', out, f'({out.stat().st_size/1e9:.2f} GB)')


/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Object B (orth): (127657, 22123)
source_b
object_a                       102460
Yan 2017 GSE92865 (10x)         13247
Yan/Kuo 2017 GSE99457 (10x)      5570
GSE145866 (10x)                  4815
Haber 2017 (Smart-Seq2)          1453
Grün 2016 (CEL-Seq)                96
Simmini 2014 GSE62784              16
Name: count, dtype: int64


saved /Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced/object_b_cross_species_orth.h5ad (1.05 GB)
